# Health Insurance Cross-Sell — Pipeline Walkthrough

> Propensity ranking to prioritize a fixed sales-call capacity.

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH)
print(df.shape)
df.head()

(8000, 12)


,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,79510,Female,22,1,19.0,1,< 1 Year,No,54559.0,152.0,87,0
1,186555,Male,57,1,18.0,0,1-2 Year,Yes,33439.0,124.0,63,0
2,158037,Female,64,1,28.0,0,1-2 Year,Yes,39327.0,122.0,45,0
3,343706,Male,52,1,41.0,1,1-2 Year,No,26536.0,124.0,161,0
4,137903,Female,21,1,7.0,0,< 1 Year,No,28416.0,152.0,92,0


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
X, y = Preprocessor().run(df)
print("features:", X.shape, "| positive rate:", round(y.mean(), 4))
X.head()

features: (8000, 10) | positive rate: 0.1226


,Age,Annual_Premium,Vintage,Region_Code,Policy_Sales_Channel,Driving_License,Previously_Insured,Gender,Vehicle_Damage,Vehicle_Age
0,22,54559.0,87,19.0,152.0,1,1,Female,No,< 1 Year
1,57,33439.0,63,18.0,124.0,1,0,Male,Yes,1-2 Year
2,64,39327.0,45,28.0,122.0,1,0,Female,Yes,1-2 Year
3,52,26536.0,161,41.0,124.0,1,1,Male,No,1-2 Year
4,21,28416.0,92,7.0,152.0,1,0,Female,No,< 1 Year


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "trained_at": "2026-06-14T03:38:40+00:00",
  "dataset": "anmolkumar/health-insurance-cross-sell-prediction",
  "data_sha256": "3b60d1072851756367817fb155e6e755ffa79509daa806720719bc395dfbea7d",
  "target": "Response",
  "problem": "propensity ranking (imbalanced binary classification)",
  "best_model": "hist_gb",
  "best_params": {
    "clf__max_leaf_nodes": 127,
    "clf__max_iter": 600,
    "clf__max_depth": 4,
    "clf__learning_rate": 0.17233109056135082,
    "clf__l2_regularization": 10.0
  },
  "cv_selection": [
    {
      "model": "hist_gb",
      "roc_auc_mean": 0.8536049944700695,
      "roc_auc_std": 0.0019503128982546402
    },
    {
      "model": "logreg",
      "roc_auc_mean": 0.8363280740060206,
      "roc_auc_std": 0.00200181812002054
    },
    {
      "model": "random_forest",
      "roc_auc_mean": 0.834665487631218,
      "roc_auc_std": 0.002352384718608383
    }
  ],
  "baseline": {
    "model": "LogisticRegression (balanced)",
    "r

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
rec = {"Age":40,"Gender":"Male","Driving_License":1,"Region_Code":28.0,"Previously_Insured":0,"Vehicle_Age":"1-2 Year","Vehicle_Damage":"Yes","Annual_Premium":30000.0,"Policy_Sales_Channel":26.0,"Vintage":150}
print("propensity:", round(pred.score_one(rec), 4))
print("top features:", pred.top_features(5))

propensity: 0.8064
top features: [{'feature': 'Previously_Insured', 'importance': 0.18336}, {'feature': 'Vehicle_Damage', 'importance': 0.05941}, {'feature': 'Policy_Sales_Channel', 'importance': 0.03416}, {'feature': 'Age', 'importance': 0.03308}, {'feature': 'Vehicle_Age', 'importance': 0.00988}]


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```